In [ ]:
!pip install rouge-score bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.9 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5cf5e24e5e2963119b8b40831105b60778b75517e0f5603f5dc85433c5dc71b6
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


#### LOADING THE DATASET

In [ ]:
import os
import torch
from rouge_score import rouge_scorer
from bert_score import score
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

folder_path = "/content/drive/MyDrive/DS310/Project/Dataset/merged_data/"

Mounted at /content/drive


In [ ]:
train_df = pd.read_csv(folder_path + "train.csv")
val_df = pd.read_csv(folder_path + "val.csv")
test_df = pd.read_csv(folder_path + "test.csv")

In [ ]:
test_df

,title,link,date_posted,headline,source,abstract,full_story,category
0,Why Some People Age Faster. And the 400 Genes ...,https://www.sciencedaily.com/releases/2025/08/...,"August 22, 2025",Why some people age faster. And the 400 genes ...,University of Colorado at Boulder,Researchers identified over 400 genes tied to ...,Scientists have uncovered more than 400 genes ...,health_medicine
1,Scientists Uncover a Multibillion-Year Epic Wr...,https://www.sciencedaily.com/releases/2024/05/...,"May 31, 2024",Scientists uncover a multibillion-year epic wr...,Tokyo Institute of Technology,Life evolved from simple geochemical processes...,The origin of life on Earth has long been a my...,fossils_ruins
2,Marfan Syndrome Increases Risk of Brain Altera...,https://www.sciencedaily.com/releases/2025/05/...,"May 16, 2025",Marfan syndrome increases risk of brain altera...,Universitat Autonoma de Barcelona,A study reveals that inflammation associated w...,A study by the Institut de Neurociències of th...,mind_brain
3,Telemedicine May Help Reduce Use of Unnecessar...,https://www.sciencedaily.com/releases/2025/02/...,"February 24, 2025",Telemedicine may help reduce use of unnecessar...,Mass General Brigham,A research team has found that telemedicine ma...,Low-value care -- medical tests and procedures...,science_society
4,"Lasers Just Unlocked a Hidden Side of Gold, Co...",https://www.sciencedaily.com/releases/2025/07/...,"July 19, 2025","Lasers just unlocked a hidden side of gold, co...",The Hebrew University of Jerusalem,Scientists have cracked a century-old physics ...,"Using only a blue laser and smart modulation, ...",matter_energy
...,...,...,...,...,...,...,...,...
1227,Simple Amino Acid Supplement Greatly Reduces A...,https://www.sciencedaily.com/releases/2025/11/...,"November 21, 2025",Simple amino acid supplement greatly reduces A...,Kindai University,Researchers discovered that the common amino a...,"New research reveals that arginine—a safe, ine...",mind_brain
1228,NaN,https://www.sciencedaily.com/releases/2025/09/...,"September 29, 2025",Black hole discovery confirms Einstein and Haw...,Simons Foundation,A fresh black hole merger detection has offere...,"When two black holes collide and merge, they r...",space_time
1229,Federal Subsidies for US Commercial Fisheries ...,https://www.sciencedaily.com/releases/2019/04/...,"April 5, 2019",Federal subsidies for US commercial fisheries ...,Duke University,A pending rule change proposed by the US Natio...,"In late 2018, the U.S. National Marine Fisheri...",business_industry
1230,Breakthrough Brain Discovery Reveals a Natural...,https://www.sciencedaily.com/releases/2025/11/...,"November 4, 2025",Breakthrough brain discovery reveals a natural...,University of Sydney,"Using powerful 7-Tesla brain imaging, research...",University of Sydney scientists have identifie...,health_medicine


#### LOAD MODEL

In [ ]:
!pip install -q bitsandbytes accelerate transformers peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 39.2 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained( model_name, torch_dtype=torch.float16, device_map="auto" )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

#### LOAD EVALUATION

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score

def evaluate_summaries(refs, preds):
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=True
    )

    r1, r2, rl = [], [], []
    for ref, pred in zip(refs, preds):
        scores = scorer.score(ref, pred)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)

    P, R, F = score(preds, refs, lang="en")

    return {
        "ROUGE-1": 100 * sum(r1) / len(r1),
        "ROUGE-2": 100 * sum(r2) / len(r2),
        "ROUGE-L": 100 * sum(rl) / len(rl),
        "BERTScore": 100 * float(F.mean())
    }

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### ZERO SHOT PROMPTING

In [ ]:
def qwen25_abstractive_summarize(article):
    article = clean_text(article)

    # auto length ~8%
    orig_len = len(article.split())
    max_new_tokens = min(int(orig_len * 0.08 * 1.3), 128)
    min_new_tokens = int(max_new_tokens * 0.4)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a news article abstractive summarization system. "
                "Produce a concise summary that preserves the key facts and main events. "
                "The summary should be approximately 8 percent of the original article length. "
                "Do not copy sentences verbatim. "
                "Do not introduce new information. "
                "Output only the summary text."
            )
        },
        {
            "role": "user",
            "content": f"Article:\n{article}"
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            do_sample=False
        )

    summary = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True
    )

    return clean_summary(summary)

In [ ]:
SYSTEM_PROMPT = (
    "You are an abstractive summarization system for English news articles. "
    "Generate a concise and coherent summary that captures the main events and key facts. "
    "The summary should be approximately 8 percent of the original article length. "
    "Do not copy sentences verbatim. "
    "Do not add new information. "
    "Output only the summary text."
)

In [ ]:
def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = t.replace("\x00", "").replace("\ufffd", "")
    return t.strip()

def auto_length(text):
    orig_len = len(text.split())
    max_len = min(int(orig_len * 0.08 * 1.3), 128)  # token approx
    min_len = int(max_len * 0.4)
    return max_len, min_len

In [ ]:
def summarize_batch_qwen(texts, batch_size=2):
    summaries = []
    num_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(num_batches):
        batch_texts = texts[i*batch_size:(i+1)*batch_size]
        print(f"\n===== BATCH {i+1}/{num_batches} =====")

        cleaned = [clean_text(t) for t in batch_texts]

        max_lens, min_lens = [], []
        for t in cleaned:
            mx, mn = auto_length(t)
            max_lens.append(mx)
            min_lens.append(mn)

        final_max_new_tokens = max(max_lens)
        final_min_new_tokens = min(min_lens)

        batch_summaries = []

        for text in cleaned:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Article:\n{text}"}
            ]

            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to(device)

            try:
                with torch.no_grad():
                    outputs = model.generate(
                        inputs,
                        max_new_tokens=final_max_new_tokens,
                        min_new_tokens=final_min_new_tokens,
                        do_sample=False,
                        temperature=0.0
                    )

                summary = tokenizer.decode(
                    outputs[0][inputs.shape[1]:],
                    skip_special_tokens=True
                )

                summary = " ".join(summary.split())
                batch_summaries.append(summary)

            except Exception as e:
                print("Generation failed:", e)
                batch_summaries.append("")

        for j, s in enumerate(batch_summaries):
            print(f"\n--- Sample {j+1} ---")
            print(s[:300], "...")

        summaries.extend(batch_summaries)

    return summaries

In [ ]:
test_texts = test_df["full_story"].astype(str).tolist()

test_df["qwen_summary"] = summarize_batch_qwen(
    test_texts,
    batch_size=2
)

metrics = evaluate_summaries(
    refs=test_df["abstract"].astype(str).tolist(),
    preds=test_df["qwen_summary"].tolist()
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



===== BATCH 1/616 =====

--- Sample 1 ---
Scientists have identified over 400 genes linked to various forms of unhealthy aging, providing insights into why some individuals age more gracefully than others. This study, published in Nature Genetics, reveals that different genetic factors contribute to distinct types of frailty, such as cognit ...

--- Sample 2 ---
Scientists have discovered that only eight new biochemical reactions are necessary to connect the dots between geochemical and biological molecules, potentially revealing previously unknown aspects of Earth's early life forms. Researchers from the Earth-Life Science Institute and Caltech modeled the ...

===== BATCH 2/616 =====

--- Sample 1 ---
A study by the Institut de Neurociències of the Universitat Autònoma de Barcelona (INc-UAB) indicates that Marfan syndrome, a genetic disorder affecting about 1 in 5,000 people, ...

--- Sample 2 ---
Telemedicine adoption appears to correlate with reduced utilization of certain low-val

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
print("\n=========== FINAL METRICS ===========")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


=========== FINAL METRICS ===========
ROUGE-1: 38.5301
ROUGE-2: 14.7961
ROUGE-L: 25.9054
BERTScore: 88.8439


In [ ]:
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(
    "/content/drive/MyDrive/DS310/Project/Results/qwen_zs_metrics.csv",
    index=False
)

# 5. Lưu toàn bộ kết quả summary ra CSV
OUTPUT_PATH = "/content/drive/MyDrive/DS310/Project/Results/qwen_zeroshot_summaries.csv"

test_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Saved summaries to:", OUTPUT_PATH)

✅ Saved summaries to: /content/drive/MyDrive/DS310/Project/Results/qwen_zeroshot_summaries.csv


#### FEW SHOT PROMPTING

In [ ]:
SYSTEM_PROMPT = (
    "You are an abstractive summarization system for English news articles. "
    "Generate a concise and coherent summary that captures the main events and key facts. "
    "The summary should be approximately 8 percent of the original article length. "
    "Do not copy sentences verbatim. "
    "Do not add new information. "
    "Output only the summary text."
)

In [ ]:
FEWSHOT_EXAMPLES = [
    # ===== Example 1 =====
    {
        "role": "user",
        "content": (
            "Article:\n"
            "A meta-analysis examined abusive supervision across multiple organizations. "
            "The study found that hostile leadership behaviors harm employee well-being and "
            "increase retaliatory actions and workplace deviance."
        )
    },
    {
        "role": "assistant",
        "content": (
            "A meta-analysis shows that abusive supervision damages employee well-being "
            "and increases retaliation and deviant workplace behavior."
        )
    },

    # ===== Example 2 =====
    {
        "role": "user",
        "content": (
            "Article:\n"
            "Researchers compared AI-generated and human-written news articles using reader surveys. "
            "The findings show automated articles are harder to understand due to poor wording "
            "and numerical explanations."
        )
    },
    {
        "role": "assistant",
        "content": (
            "A reader survey finds AI-generated news articles less comprehensible than "
            "human-written ones, mainly due to weak word choice and handling of numbers."
        )
    }
]

In [ ]:
def qwen25_abstractive_summarize_fs(article):
    article = clean_text(article)

    orig_len = len(article.split())
    max_new_tokens = min(int(orig_len * 0.08 * 1.3), 128)
    min_new_tokens = int(max_new_tokens * 0.4)

    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + FEWSHOT_EXAMPLES
        + [{"role": "user", "content": f"Article:\n{article}"}]
    )

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    prompt_len = input_ids.shape[1]
    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            do_sample=False
        )

    summary = tokenizer.decode(
        outputs[0][prompt_len:],
        skip_special_tokens=True
    )

    return clean_summary(summary)

In [ ]:
def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = t.replace("\x00", "").replace("\ufffd", "")
    return t.strip()

def auto_length(text):
    orig_len = len(text.split())
    max_len = min(int(orig_len * 0.08 * 1.3), 128)  # token approx
    min_len = int(max_len * 0.4)
    return max_len, min_len

In [ ]:
def summarize_batch_qwen_fs(texts, batch_size=2):
    summaries = []
    num_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(num_batches):
        batch_texts = texts[i*batch_size:(i+1)*batch_size]
        print(f"\n===== BATCH {i+1}/{num_batches} =====")

        cleaned = [clean_text(t) for t in batch_texts]

        max_lens, min_lens = [], []
        for t in cleaned:
            mx, mn = auto_length(t)
            max_lens.append(mx)
            min_lens.append(mn)

        final_max_new_tokens = max(max_lens)
        final_min_new_tokens = min(min_lens)

        batch_summaries = []

        for text in cleaned:
            messages = (
                [{"role": "system", "content": SYSTEM_PROMPT}]
                + FEWSHOT_EXAMPLES
                + [{"role": "user", "content": f"Article:\n{text}"}]
            )

            input_ids = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to(device)

            prompt_len = input_ids.shape[1]
            attention_mask = (input_ids != tokenizer.pad_token_id).long()

            try:
                with torch.no_grad():
                    outputs = model.generate(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        max_new_tokens=final_max_new_tokens,
                        min_new_tokens=final_min_new_tokens,
                        do_sample=False
                    )

                summary = tokenizer.decode(
                    outputs[0][prompt_len:],
                    skip_special_tokens=True
                )

                batch_summaries.append(" ".join(summary.split()))

            except Exception as e:
                print("Generation failed:", e)
                batch_summaries.append("")

        for j, s in enumerate(batch_summaries):
            print(f"\n--- Sample {j+1} ---")
            print(s[:300], "...")

        summaries.extend(batch_summaries)

    return summaries

In [ ]:
test_texts = test_df["full_story"].astype(str).tolist()

test_df["qwen_summary_fs"] = summarize_batch_qwen_fs(
    test_texts,
    batch_size=2
)

metrics = evaluate_summaries(
    refs=test_df["abstract"].astype(str).tolist(),
    preds=test_df["qwen_summary_fs"].tolist()
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



===== BATCH 1/616 =====

--- Sample 1 ---
Scientists have identified more than 400 genes linked to different forms of unhealthy aging, providing insights into why some individuals age better than others. The study, published in Nature Genetics, reveals that various genetic factors contribute to different types of frailty, such as cognitive  ...

--- Sample 2 ---
Researchers from the Earth-Life Science Institute and Caltech modeled the history of biochemistry to understand how much of Earth's life history may be lost. They discovered that the presence of adenosine triphosphate (ATP), crucial for cellular energy, creates a bottleneck in metabolic pathways. By ...

===== BATCH 2/616 =====

--- Sample 1 ---
A study by researchers at the Institut de Neurociències of the Universitat Autònoma de Barcelona (INc-UAB) in collaboration with other institutions has found that inflammation linked to Marfan syndrome increases the risk of neurological diseases ...

--- Sample 2 ---
Research by a team

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
print("\n=========== FINAL METRICS ===========")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


=========== FINAL METRICS ===========
ROUGE-1: 38.6210
ROUGE-2: 14.7280
ROUGE-L: 25.5586
BERTScore: 88.7621


In [ ]:
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(
    "/content/drive/MyDrive/DS310/Project/Results/qwen_fs_metrics.csv",
    index=False
)

# 5. Lưu toàn bộ kết quả summary ra CSV
OUTPUT_PATH = "/content/drive/MyDrive/DS310/Project/Results/qwen_fewshot_summaries.csv"

test_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Saved summaries to:", OUTPUT_PATH)

✅ Saved summaries to: /content/drive/MyDrive/CS406/qwen_fewshot_summaries.csv


# Factual consistency

In [ ]:
import pandas as pd


In [ ]:
PATH = "/content/drive/MyDrive/DS310/Project/Results/"

test_df_zs = pd.read_csv(PATH + "qwen_zeroshot_summaries.csv")
test_df_fs = pd.read_csv(PATH + "qwen_fewshot_summaries.csv")

In [ ]:
from transformers import pipeline
import torch

nli = pipeline(
    "text-classification",
    model="roberta-large-mnli",
    return_all_scores=False,
    device=0 if torch.cuda.is_available() else -1
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [ ]:
def factual_consistency_score(inputs, summaries, max_chars=600):
    labels = []
    scores = []

    for inp, summ in zip(inputs, summaries):
        if not summ or not inp:
            labels.append("EMPTY")
            scores.append(0.0)
            continue

        premise = inp[:max_chars]
        hypothesis = summ

        try:
            result = nli({
                "text": premise,
                "text_pair": hypothesis
            })

            if isinstance(result, list):
                result = result[0]

            labels.append(result["label"])
            scores.append(result["score"])

        except Exception as e:
            print("NLI ERROR:", repr(e))
            labels.append("ERROR")
            scores.append(0.0)

    return labels, scores

In [ ]:
print(nli({
    "text": test_df_zs["full_story"].iloc[0][:600],
    "text_pair": test_df_zs["qwen_summary"].iloc[0]
}))

{'label': 'NEUTRAL', 'score': 0.9797462821006775}


In [ ]:
labels_zs, scores_zs = factual_consistency_score(
    test_df_zs["full_story"].astype(str).tolist(),
    test_df_zs["qwen_summary"].astype(str).tolist()
)

test_df_zs["nli_label"] = labels_zs
test_df_zs["nli_score"] = scores_zs

In [ ]:
labels_fs, scores_fs = factual_consistency_score(
    test_df_fs["full_story"].astype(str).tolist(),
    test_df_fs["qwen_summary_fs"].astype(str).tolist()
)

test_df_fs["nli_label"] = labels_fs
test_df_fs["nli_score"] = scores_fs

In [ ]:
print("Qwen Zero-shot NLI")
print(test_df_zs["nli_label"].value_counts(normalize=True) * 100)

print("\nQwen Few-shot NLI")
print(test_df_fs["nli_label"].value_counts(normalize=True) * 100)

Qwen Zero-shot NLI
nli_label
NEUTRAL          83.847403
ENTAILMENT       14.853896
CONTRADICTION     1.298701
Name: proportion, dtype: float64

Qwen Few-shot NLI
nli_label
NEUTRAL          87.743506
ENTAILMENT       10.876623
CONTRADICTION     1.379870
Name: proportion, dtype: float64


# Compression & Coverage

In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download("punkt")

from nltk.util import ngrams

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
def compression_ratio(input_text, summary):
    if not input_text or not summary:
        return 0.0
    return len(summary.split()) / max(len(input_text.split()), 1)


def coverage(input_text, summary):
    if not input_text or not summary:
        return 0.0
    input_words = set(input_text.lower().split())
    summary_words = summary.lower().split()
    return sum(w in input_words for w in summary_words) / len(summary_words)


def redundancy(summary):
    if not summary:
        return 0.0
    words = summary.lower().split()
    if len(words) < 2:
        return 0.0
    bigrams = list(zip(words, words[1:]))
    return 1 - len(set(bigrams)) / max(len(bigrams), 1)


def analyze_style(df, col):
    return {
        "compression": df.apply(
            lambda x: compression_ratio(x["full_story"], x[col]), axis=1
        ).mean(),
        "coverage": df.apply(
            lambda x: coverage(x["full_story"], x[col]), axis=1
        ).mean(),
        "redundancy": df[col].apply(redundancy).mean()
    }

In [ ]:
style_qwen_zs = analyze_style(test_df_zs, "qwen_summary")
style_qwen_fs = analyze_style(test_df_fs, "qwen_summary_fs")

In [ ]:
style_df = pd.DataFrame.from_dict(
    {
        "Qwen-ZeroShot": style_qwen_zs,
        "Qwen-FewShot": style_qwen_fs
    },
    orient="index"
)

print(style_df)

               compression  coverage  redundancy
Qwen-ZeroShot     0.104292  0.749223    0.010737
Qwen-FewShot      0.102749  0.762336    0.010519
